# P2 · Phân nhóm & tổng hợp — Bước 2–3 · 9

**Input (đọc):** data/processed/data_long.parquet, common/segment_labels.parquet

**Output (bạn phải tạo, đúng format):** EDA + 2 bảng báo cáo (balanced & frequency-weighted), bảng thắng theo nhóm (bước 9)

**Code mẫu chạy được:** mở `notebooks/M5_reference.ipynb` phần **§2** — copy các cell sang đây rồi mở rộng.

> Quy tắc vàng: không sửa `common/`; không shuffle; tôn trọng trục thời gian; ML dùng lag ≥ 28. Forecast lưu vào `data/outputs/forecasts/<model>.parquet` với cột: `series_id,date,d_int,quantile_level,value,model`.

## Cell setup dùng chung (chạy trước) — nạp tham số cố định + dữ liệu
Các cell dưới đây GIỐNG NHAU ở mọi notebook, đảm bảo cả nhóm dùng chung 1 cấu hình & dữ liệu.

In [ ]:
import sys, subprocess, importlib, time, warnings, os, json, math, random
# --- 0.2 Import lõi + dò package nào dùng được ------------------------------
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
np.random.seed(42); random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4); plt.rcParams["axes.grid"] = True

def can_import(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

AVAILABLE = {
    "pyarrow":      can_import("pyarrow"),
    "lightgbm":     can_import("lightgbm"),
    "statsforecast":can_import("statsforecast"),
    "chronos":      can_import("chronos"),
    "torch":        can_import("torch"),
    "ttm":          can_import("tsfm_public"),
}
try:
    import torch
    AVAILABLE["cuda"] = torch.cuda.is_available()
except Exception:
    AVAILABLE["cuda"] = False

print("Package/phần cứng dùng được:")
for k, v in AVAILABLE.items():
    print(f"  {'✓' if v else '✗'} {k}: {v}")
PARQUET_ENGINE = "pyarrow" if AVAILABLE["pyarrow"] else None

In [ ]:
# --- Nạp THAM SỐ CỐ ĐỊNH từ common/config.yaml (nguồn chân lý duy nhất) -------
import yaml
from pathlib import Path
_c = yaml.safe_load(open("common/config.yaml", encoding="utf-8"))
CONFIG = dict(
    SEED=_c["seed"], HORIZON=_c["horizon"],
    SAMPLE_PER_SEGMENT=_c.get("sampling", {}).get("cap", 1000),
    TRAIN_END=_c["split"]["train_end"], VAL_END=_c["split"]["val_end"],
    TEST_END=_c["split"]["test_end"],
    QUANTILES=_c["quantiles"], ADI_TH=_c["sbc"]["adi_threshold"],
    CV2_TH=_c["sbc"]["cv2_threshold"],
    CHRONOS_MODEL=_c["foundation"]["chronos_model"],
    CHRONOS_CONTEXT=_c["foundation"]["chronos_context"],
    TTM_MODEL=_c["foundation"]["ttm_model"],
    TTM_CONTEXT=_c["foundation"]["ttm_context"],
)
CONFIG["TEST_START"] = CONFIG["VAL_END"] + 1
Q = CONFIG["QUANTILES"]
RAW = Path("data/raw"); OUT = Path("data/outputs"); FC = OUT / "forecasts"
OUT.mkdir(parents=True, exist_ok=True); FC.mkdir(parents=True, exist_ok=True)
PARQUET_ENGINE = "pyarrow"
def save_parquet(df, path):
    path = Path(path); df.to_parquet(path, index=False); return path
print("CONFIG nạp từ common/config.yaml:", {k: CONFIG[k] for k in ["SEED","HORIZON","TRAIN_END","VAL_END","TEST_END"]})

In [ ]:
# --- Nạp DỮ LIỆU CHUNG: long (P1 đã sinh) + mẫu + nhãn + calendar/giá ---------
import numpy as np, pandas as pd
calendar = pd.read_csv(RAW / "calendar.csv")
prices   = pd.read_csv(RAW / "sell_prices.csv")

_long_path = Path("data/processed/data_long.parquet")
assert _long_path.exists(), ("Chưa có data_long. Hãy chạy notebooks/P1.ipynb trước "
                             "(hoặc lấy từ người P1) để sinh data/processed/data_long.parquet.")
long = pd.read_parquet(_long_path)
long["date"] = pd.to_datetime(long["date"])

sample_ids = pd.read_csv("common/sample_ids.csv")["series_id"].tolist()
seg_labels = pd.read_parquet("common/segment_labels.parquet")

# schema ngày (train/val/test) cho cửa sổ gốc
schema = dict(
    train_days=[f"d_{i}" for i in range(1, CONFIG["TRAIN_END"]+1)],
    val_days=[f"d_{i}" for i in range(CONFIG["TRAIN_END"]+1, CONFIG["VAL_END"]+1)],
    test_days=[f"d_{i}" for i in range(CONFIG["VAL_END"]+1, CONFIG["TEST_END"]+1)],
)
# insample (<= val_end) dùng cho model cổ điển & FM
insample = long[long.d_int <= CONFIG["VAL_END"]].sort_values(["series_id","d_int"])
print(f"long: {long.shape} | mẫu: {len(sample_ids)} chuỗi | nhãn: {seg_labels.shape}")

In [ ]:
# --- 2.5 Bảng actuals cho test (dùng cho đánh giá) --------------------------
# actual test: các ngày d_TEST_START..d_TEST_END của chuỗi mẫu
test_days = schema["test_days"]
actual_test = (long[long.d.isin(test_days)]
               [["series_id","date","d_int","sales","segment"]]
               .rename(columns={"sales":"actual"})
               .reset_index(drop=True))
# train history (long) để tính scale RMSSE/pinball
train_hist = long[long.d_int <= CONFIG["VAL_END"]][["series_id","d_int","sales"]].copy()
print("actual_test:", actual_test.shape, "| train_hist:", train_hist.shape)

In [ ]:
# --- 3.1 Tiện ích scale (RMSSE & pinball) + trọng số theo chuỗi -------------
def _scales(y):
    '''scale MSE & MAE cua naive-1 tren insample, tinh tu lan ban dau tien (kieu M5).'''
    nz = np.nonzero(y)[0]
    if len(nz) < 2:
        return np.nan, np.nan
    y = y[nz[0]:]
    d = np.diff(y)
    if len(d) == 0:
        return np.nan, np.nan
    return float(np.mean(d**2)), float(np.mean(np.abs(d)))

scale_rmsse, scale_mae = {}, {}
for sid, g in train_hist.sort_values("d_int").groupby("series_id"):
    s2, s1 = _scales(g["sales"].to_numpy(dtype=np.float64))
    scale_rmsse[sid], scale_mae[sid] = s2, s1

# trọng số WSPL = doanh số 28 ngày insample cuối (kiểu M5 uncertainty)
last28 = long[(long.d_int > CONFIG["VAL_END"]-28) & (long.d_int <= CONFIG["VAL_END"])].copy()
last28["dollar"] = last28["sales"] * last28["sell_price"].fillna(0)
weight = last28.groupby("series_id")["dollar"].sum()
weight = (weight / weight.sum()).to_dict()   # chuẩn hoá tổng = 1
print(f"Đã tính scale cho {len(scale_rmsse)} chuỗi. Ví dụ scale_mae: "
      f"{list(scale_mae.items())[0]}")

In [ ]:
# --- 3.2 Dispersion NB theo chuỗi (từ insample) -----------------------------
mv = train_hist.groupby("series_id")["sales"].agg(["mean","var"])
def _r(row):
    m, v = row["mean"], row["var"]
    if v > m and m > 0:
        return m*m / (v - m)      # r hữu hạn -> NB
    return np.inf                 # -> Poisson
disp_r = mv.apply(_r, axis=1).to_dict()

def point_to_quantiles(point_df, name):
    '''point_df: series_id,date,d_int,value(mean) -> long 9 quantile theo NB/Poisson.'''
    out = []
    pf = point_df.copy()
    pf["value"] = pf["value"].clip(lower=0).astype(float)
    pf["r"] = pf["series_id"].map(disp_r).fillna(np.inf)
    mu = pf["value"].to_numpy()
    r  = pf["r"].to_numpy()
    finite = np.isfinite(r) & (r > 0)
    for u in Q:
        qv = np.empty_like(mu)
        # NB: mean=mu, size=r  -> p = r/(r+mu)
        if finite.any():
            rr = r[finite]; mm = mu[finite]
            p = rr / (rr + np.where(mm > 0, mm, 1e-9))
            qv[finite] = stats.nbinom.ppf(u, rr, p)
        if (~finite).any():
            qv[~finite] = stats.poisson.ppf(u, mu[~finite])
        block = pf[["series_id","date","d_int"]].copy()
        block["quantile_level"] = u
        block["value"] = np.nan_to_num(qv, nan=0.0)
        out.append(block)
    res = pd.concat(out, ignore_index=True)
    res["model"] = name
    return res[["series_id","date","d_int","quantile_level","value","model"]]
print("Sẵn sàng point_to_quantiles(). Ví dụ disp_r:", list(disp_r.items())[0])

In [ ]:
# --- 3.3 evaluate(): RMSSE, SPL/WSPL, coverage theo từng chuỗi --------------
COV_PAIRS = [(0.25,0.75,0.50),(0.165,0.835,0.67),(0.025,0.975,0.95),(0.005,0.995,0.99)]

def evaluate(fdf, name):
    '''fdf: long forecast (series_id,d_int,quantile_level,value). Tra per-series metrics.'''
    piv = (fdf.pivot_table(index=["series_id","d_int"], columns="quantile_level", values="value")
              .reset_index())
    m = piv.merge(actual_test[["series_id","d_int","actual","segment"]],
                  on=["series_id","d_int"], how="inner")
    rows = []
    for sid, g in m.groupby("series_id"):
        a = g["actual"].to_numpy(dtype=float)
        p50 = g[0.5].to_numpy(dtype=float) if 0.5 in g else np.full_like(a, np.nan)
        sr, sm = scale_rmsse.get(sid, np.nan), scale_mae.get(sid, np.nan)
        rmsse = np.sqrt(np.mean((a-p50)**2)/sr) if (sr and sr > 0) else np.nan
        spls = []
        for u in Q:
            if u in g:
                q = g[u].to_numpy(dtype=float)
                pin = np.mean(np.maximum(u*(a-q), (u-1)*(a-q)))
                spls.append(pin/sm if (sm and sm > 0) else np.nan)
        spl = np.nanmean(spls) if spls else np.nan
        covs = {}
        for lo, hi, nom in COV_PAIRS:
            if lo in g and hi in g:
                inside = ((a >= g[lo].to_numpy()) & (a <= g[hi].to_numpy())).mean()
                covs[f"cov{int(nom*100)}"] = inside
        rows.append(dict(series_id=sid, segment=g["segment"].iloc[0], model=name,
                         RMSSE=rmsse, SPL=spl, weight=weight.get(sid, np.nan), **covs))
    return pd.DataFrame(rows)
print("evaluate() sẵn sàng.")

## Việc của P2 (bắt đầu bên dưới)

In [ ]:
# --- 2.3 EDA trực quan -------------------------------------------------------
fig, ax = plt.subplots(2, 2, figsize=(12, 9))

# (a) số chuỗi theo nhóm
summary.n_series.plot.bar(ax=ax[0,0], color=["#4c72b0","#dd8452","#55a868","#c44e52"])
ax[0,0].set_title("(a) Số chuỗi theo nhóm demand"); ax[0,0].set_ylabel("số chuỗi")

# (b) tỷ lệ zero theo nhóm
data = [seg_df.loc[seg_df.segment==s, "zero_rate"].values
        for s in ["smooth","erratic","intermittent","lumpy"]]
ax[0,1].boxplot(data, labels=["smooth","erratic","interm.","lumpy"], showfliers=False)
ax[0,1].set_title("(b) Tỷ lệ ngày = 0 theo nhóm"); ax[0,1].set_ylabel("zero rate")

# (c) scatter ADI vs CV^2 (mẫu để nhẹ) + ngưỡng
s = seg_df[np.isfinite(seg_df.ADI)].sample(min(4000, len(seg_df)), random_state=1)
colors = {"smooth":"#4c72b0","erratic":"#dd8452","intermittent":"#55a868","lumpy":"#c44e52"}
for seg in colors:
    d = s[s.segment==seg]
    ax[1,0].scatter(d.ADI, d.CV2, s=4, alpha=0.3, label=seg, color=colors[seg])
ax[1,0].axvline(adi_th, ls="--", c="k", lw=1); ax[1,0].axhline(cv2_th, ls="--", c="k", lw=1)
ax[1,0].set_xlim(0, 8); ax[1,0].set_ylim(0, 4)
ax[1,0].set_xlabel("ADI"); ax[1,0].set_ylabel("CV²"); ax[1,0].set_title("(c) Phân nhóm SBC")
ax[1,0].legend(markerscale=3, fontsize=8)

# (d) mùa vụ theo thứ trong tuần (tổng sales toàn tập, dùng calendar)
wd = calendar[["d","wday"]].merge(
        sales[schema['train_days']].sum(axis=0).rename("tot").reset_index().rename(columns={"index":"d"}),
        on="d")
wd.groupby("wday").tot.mean().plot.bar(ax=ax[1,1], color="#8172b3")
ax[1,1].set_title("(d) Mùa vụ theo wday (1=Sat..7=Fri)"); ax[1,1].set_ylabel("TB sales/ngày")

plt.tight_layout(); plt.show()